In [74]:
import json

In [94]:
from __future__ import annotations

import csv
import hashlib
import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np


# ----------------------------
# Helpers
# ----------------------------

def iter_attack_json_files(folder: Path) -> Iterable[Path]:
    """Yield all *.attack.json files recursively under folder."""
    if not folder.exists():
        return
    yield from folder.rglob("*.attack.json")


def load_json(path: Path) -> Optional[dict]:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def extract_sanitized_text(obj: dict) -> Optional[str]:
    """Extract sanitized text from known keys."""
    for k in ("sanitized_txt", "sanitized_text", "sanitized"):
        v = obj.get(k)
        if isinstance(v, str) and v.strip():
            return v
    return None


def normalize_for_fingerprint(text: str) -> str:
    """Normalize before hashing."""
    text = text.strip()
    text = " ".join(text.split())   # collapse whitespace
    text = text.lower()
    return text


def sha256_hex(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()


@dataclass(frozen=True)
class Occurrence:
    folder: str
    file_path: str
    row_index: Optional[int]
    text_len: int


# ----------------------------
# Core logic
# ----------------------------

def build_index(folders: List[Path], min_chars: int = 200) -> Tuple[Dict[str, List[Occurrence]], Dict[str, Dict]]:
    """
    Returns:
      - fingerprint -> list of Occurrence
      - fingerprint -> metadata dict (snippet, text_len)
    """
    index: Dict[str, List[Occurrence]] = defaultdict(list)
    meta: Dict[str, Dict] = {}

    for folder in folders:
        folder_label = folder.name  # label for outputs
        for f in iter_attack_json_files(folder):
            if not f.is_file():
                continue

            obj = load_json(f)
            if not obj:
                continue

            s = extract_sanitized_text(obj)
            if not s:
                continue

            s_norm = normalize_for_fingerprint(s)
            if len(s_norm) < min_chars:
                continue

            fp = sha256_hex(s_norm)

            row_index = obj.get("row_index")
            if not isinstance(row_index, int):
                row_index = None

            index[fp].append(
                Occurrence(
                    folder=folder_label,
                    file_path=str(f),
                    row_index=row_index,
                    text_len=len(s_norm),
                )
            )

            if fp not in meta:
                meta[fp] = {
                    "snippet": (s_norm[:240] + "…") if len(s_norm) > 240 else s_norm,
                    "text_len": len(s_norm),
                }

    return index, meta


def find_cross_folder_duplicates(index: Dict[str, List[Occurrence]]) -> Dict[str, List[Occurrence]]:
    """Keep only groups that appear in 2+ folders."""
    out = {}
    for fp, occs in index.items():
        if len({o.folder for o in occs}) >= 2:
            out[fp] = occs
    return out


def build_folder_overlap_matrix(dups: Dict[str, List[Occurrence]]) -> Dict[Tuple[str, str], int]:
    """Count how many duplicate groups are shared between folder pairs."""
    counts: Dict[Tuple[str, str], int] = defaultdict(int)
    for fp, occs in dups.items():
        present = sorted({o.folder for o in occs})
        for i in range(len(present)):
            for j in range(i + 1, len(present)):
                counts[(present[i], present[j])] += 1
    return counts


def write_duplicates_report_json(out_path: Path, dups: Dict[str, List[Occurrence]], meta: Dict[str, Dict]) -> None:
    payload = []
    for fp, occs in sorted(dups.items(), key=lambda x: (-len(x[1]), x[0])):
        by_folder: Dict[str, List[dict]] = defaultdict(list)
        for o in occs:
            by_folder[o.folder].append({
                "file_path": o.file_path,
                "row_index": o.row_index,
                "text_len": o.text_len,
            })

        payload.append({
            "fingerprint_sha256": fp,
            "folders_involved": sorted(by_folder.keys()),
            "total_occurrences": len(occs),
            "per_folder_counts": {k: len(v) for k, v in by_folder.items()},
            "snippet": meta.get(fp, {}).get("snippet", ""),
            "text_len": meta.get(fp, {}).get("text_len", None),
            "occurrences": dict(by_folder),
        })

    out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


# ----------------------------
# Jupyter-friendly "main"
# ----------------------------

def run_dedup(
    folder_paths: List[str],
    out_dir: str = "dedup_out",
    min_chars: int = 200
) -> None:
    folders = [Path(p).expanduser().resolve() for p in folder_paths]
    out_dir = Path(out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    # Basic validation
    missing = [str(p) for p in folders if not p.exists()]
    if missing:
        print("⚠️ These folders do not exist:")
        for m in missing:
            print("  -", m)
        # still continue with those that exist
        folders = [p for p in folders if p.exists()]
        if not folders:
            return

    # Scan
    total_files = sum(1 for folder in folders for _ in iter_attack_json_files(folder))
    index, meta = build_index(folders, min_chars=min_chars)
    dups = find_cross_folder_duplicates(index)

    print(f"Scanned folders: {len(folders)}")
    print(f"Total *.attack.json files found: {total_files}")
    print(f"Unique fingerprints (>= {min_chars} chars): {len(index)}")
    print(f"Cross-folder duplicate groups: {len(dups)}")

    # Aggregate "where and how many"
    dup_occ_by_folder = defaultdict(int)
    for fp, occs in dups.items():
        for o in occs:
            dup_occ_by_folder[o.folder] += 1

    if dups:
        print("\nDuplicate occurrences per folder (counting every occurrence in cross-folder groups):")
        for k, v in sorted(dup_occ_by_folder.items(), key=lambda x: -x[1]):
            print(f"  - {k}: {v}")

        pair_counts = build_folder_overlap_matrix(dups)
        # folder_labels = [p.name for p in folders]

        report_json = out_dir / "duplicates_report.json"


        write_duplicates_report_json(report_json, dups, meta)


        print(f"\nWrote:")
        print(f"  - {report_json}")

    else:
        print("\nNo cross-folder duplicates found (with current settings).")


In [113]:
folder_paths = [
    "/home/simonettos/thijs/data_augmentatio_stefano/file_with_ids_eset/file_with_ids_eset",
    "/home/simonettos/thijs/data_augmentatio_stefano/file_with_ids_aptnotes",
    "/home/simonettos/thijs/data_augmentatio_stefano/file_with_ids_apt_cybercriminals",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/mitre_rep",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mapedia",
    "/home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt"
]

run_dedup(folder_paths, out_dir="dedup_out", min_chars=200)


Scanned folders: 6
Total *.attack.json files found: 4540
Unique fingerprints (>= 200 chars): 4051
Cross-folder duplicate groups: 19

Duplicate occurrences per folder (counting every occurrence in cross-folder groups):
  - mitre_rep: 14
  - files_with_ids_mapedia: 14
  - file_with_ids_aptnotes: 5
  - file_with_ids_apt_cybercriminals: 5

Wrote:
  - /home/simonettos/thijs/classification/classification/dedup_out/duplicates_report.json


In [149]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple


# ----------------------------
# Helpers
# ----------------------------

def iter_attack_json_files(folder: Path) -> Iterable[Path]:
    yield from folder.rglob("*.attack.json")


def load_json(path: Path) -> Optional[dict]:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def extract_id_list(obj: dict) -> List[str]:
    """
    Prefer union list if available.
    Fallbacks keep compatibility with older outputs.
    """
    for k in ("ID_lis t_union", "ID_list_from_text","ID_list"):
        v = obj.get(k)
        if isinstance(v, list) and v:
            # keep only non-empty strings
            out = []
            for x in v:
                if isinstance(x, str):
                    x = x.strip()
                    if x:
                        out.append(x)
            if out:
                return out
    return []


def is_ttp_id(x: str) -> bool:
    """
    Keep technique IDs only, excluding tactics (TA****) by default.
    Adjust if you want to include TA**** too.
    """
    x = x.strip()
    return x.startswith("T") and not x.startswith("TA")


# ----------------------------
# Core counting
# ----------------------------

@dataclass
class FolderTTPStats:
    folder: str
    num_files: int
    total_ttp_mentions: int         # sum over files of number of TTPs in that file
    unique_ttps: int                # unique TTP IDs across folder
    ttp_file_frequency: Counter     # counts: in how many files each TTP appears


def compute_ttp_stats_for_folder(folder: Path) -> FolderTTPStats:
    folder_label = folder.name
    num_files = 0
    total_mentions = 0
    ttp_file_freq = Counter()
    unique_set = set()

    for f in iter_attack_json_files(folder):
        if not f.is_file():
            continue
        obj = load_json(f)
        if not obj:
            continue

        num_files += 1
        ids = extract_id_list(obj)

        # Keep only technique-like IDs, and de-dup within file for "file frequency"
        ttps = [x for x in ids if isinstance(x, str) and is_ttp_id(x)]
        ttps_unique_in_file = set(ttps)

        total_mentions += len(ttps)             # counts per-file list length (mentions)
        unique_set.update(ttps_unique_in_file)  # folder-wide unique
        ttp_file_freq.update(ttps_unique_in_file)  # in how many files each appears

    return FolderTTPStats(
        folder=folder_label,
        num_files=num_files,
        total_ttp_mentions=total_mentions,
        unique_ttps=len(unique_set),
        ttp_file_frequency=ttp_file_freq,
    )


def run_ttp_counts(folder_paths: List[str], top_k: int = 20) -> Dict[str, FolderTTPStats]:
    folders = [Path(p).expanduser().resolve() for p in folder_paths]
    stats_by_folder: Dict[str, FolderTTPStats] = {}

    missing = [str(p) for p in folders if not p.exists()]
    if missing:
        print("⚠️ These folders do not exist:")
        for m in missing:
            print("  -", m)

    folders = [p for p in folders if p.exists()]
    if not folders:
        return {}

    for folder in folders:
        s = compute_ttp_stats_for_folder(folder)
        stats_by_folder[s.folder] = s

        print(f"\n📁 {s.folder}")
        print(f"  Files scanned:         {s.num_files}")
        print(f"  Total TTP mentions:    {s.total_ttp_mentions}")
        print(f"  Unique TTP IDs:        {s.unique_ttps}")

        if top_k > 0 and s.ttp_file_frequency:
            print(f"  Top {top_k} TTPs by #files mentioned in:")
            for ttp, c in s.ttp_file_frequency.most_common(top_k):
                print(f"    - {ttp}: {c}")

    return stats_by_folder


In [150]:
folder_paths = [
    "/home/simonettos/thijs/data_augmentatio_stefano/file_with_ids_eset/file_with_ids_eset",
    "/home/simonettos/thijs/data_augmentatio_stefano/file_with_ids_aptnotes",
    "/home/simonettos/thijs/data_augmentatio_stefano/file_with_ids_apt_cybercriminals",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/mitre_rep",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mapedia",
    "/home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt"
]

stats = run_ttp_counts(folder_paths, top_k=25)



📁 file_with_ids_eset
  Files scanned:         148
  Total TTP mentions:    3170
  Unique TTP IDs:        408
  Top 25 TTPs by #files mentioned in:
    - T1041: 81
    - T1140: 74
    - T1082: 69
    - T1083: 58
    - T1071.001: 57
    - T1587.001: 57
    - T1106: 51
    - T1113: 51
    - T1057: 49
    - T1573.001: 47
    - T1005: 46
    - T1027: 44
    - T1059.003: 40
    - T1204.002: 36
    - T1112: 33
    - T1095: 33
    - T1547.001: 32
    - T1583.001: 30
    - T1132.001: 30
    - T1071: 29
    - T1016: 29
    - T1033: 27
    - T1036.005: 27
    - T1583.004: 26
    - T1105: 26

📁 file_with_ids_aptnotes
  Files scanned:         45
  Total TTP mentions:    1037
  Unique TTP IDs:        336
  Top 25 TTPs by #files mentioned in:
    - T1082: 22
    - T1059.003: 17
    - T1033: 16
    - T1140: 16
    - T1041: 16
    - T1027: 15
    - T1057: 15
    - T1083: 14
    - T1059.001: 14
    - T1071.001: 13
    - T1016: 11
    - T1105: 11
    - T1003: 11
    - T1204.002: 11
    - T1047: 11
    -